# First-minute replay

A 2D replay of the tracking, like the starter's animation but with what our pipeline sees and
decides:

- every tracked player, numbered by track slot (a slot is not a person)
- the ball where it is tracked (solid), where we filled a short gap (ring), or nothing where it is lost
- detected touches as a yellow flash
- our label on each frame, and a strip along the bottom showing the whole minute

It is written as an HTML page in `cache/`, which is not committed. Open it in any browser, or see it
below.

**The replay of the provided match contains match data.** Keep it out of cloud AI.

---

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys
sys.path[:0] = [os.path.abspath(p) for p in ('..', '../src', '../..', '../../starter', '../../evaluation')]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML, display
import html
import animate, external, degrade, model, parse, run, stage2
from games import GAME, half_video_blocks


def show(path, height=900):
    with open(path, encoding='utf-8') as handle:
        page = handle.read()
    display(HTML('<iframe srcdoc="%s" width="100%%" height="%d" style="border:0"></iframe>'
                 % (html.escape(page), height)))
    print('also saved to', os.path.abspath(path))

## 1. The provided match

`START` is a video frame and `SECONDS` the length. The first half kicks off on frame 11185 (from
`DATA.md`). Try other moments too, such as 113742 for the second-half kickoff.

In [ ]:
START, SECONDS = 11185, 60

data = parse.load()
blocks = half_video_blocks(GAME)
ours = run.predict(data, blocks, model.load(), stage2.load())
path = animate.write('../cache/replay_first_minute.html', data, blocks, START, SECONDS,
                     {'ours': ours}, 'Barça vs Athletic, from frame %d' % START)
show(path)

## 2. An open match, with the true labels

The same view on a damaged open match, with the answer strip above ours. This is the quickest way
to see what kind of mistake the pipeline makes, and it is safe to share.

In [ ]:
match_id = external.match_ids()[0]
open_data = external.load(match_id)
open_blocks = [tuple(block) for block in open_data['blocks'].tolist()]
damaged = degrade.degrade(open_data, degrade.load_profile(), 0)
predicted = run.predict(damaged, open_blocks, model.load(model.cv_path(match_id)),
                        stage2.load(stage2.cv_path(match_id)))
path = animate.write('../cache/replay_open_match.html', damaged, open_blocks, open_blocks[0][1], 60,
                     {'truth': open_data['labels'], 'ours': predicted},
                     'Open match %s, damaged' % match_id)
show(path)